<img src="../assets/logo-banner.png" alt="radar-datatree — Cloud-native, time-aware weather radar datasets" width="800" class="only-light">
<img src="../assets/logo-banner-dark.png" alt="radar-datatree — Cloud-native, time-aware weather radar datasets" width="800" class="only-dark">

# radar datatree: Cloud-native, time-aware weather radar datasets
---

You want to look at how a Chicago-area storm evolved on a given date. The traditional path: download multiple gigabytes of NEXRAD Level II files, decode them, georeference each sweep, and stitch them together. **This notebook does the same end-to-end, streaming straight from the cloud — connect, load, slice, georeference, plot.**

## Why a DataTree?

Weather radar volumes are organized hierarchically: a Volume Coverage Pattern (VCP) cycles through several 360° sweeps at increasing elevation angles, repeating every 4–10 minutes. Different VCPs have different shapes, which is exactly what `xarray.DataTree` is built for.

- **Hierarchical** — every VCP × sweep combination is a node you navigate (`dt["VCP-12/sweep_0"]`), not a separate file you parse.
- **Time-indexed** — all scans share a `vcp_time` dimension, so `.sel(vcp_time=...)` replaces the file-iteration loop.
- **Lazy** — opening the full archive fetches metadata only (~MB). Variables stream from object storage on demand.

For architecture and scaling benchmarks, see *Ladino-Rincón et al.* (2026, in preparation). Unfamiliar acronyms (VCP, polarimetric, ZDR, RHOHV, ...) are all defined in the [glossary](glossary).

## radar datatree in practice

Let's see how the radar datatree looks in practice.

In [ ]:
# Colab bootstrap — runs only on Google Colab; no-op locally and in CI.
import sys

if "google.colab" in sys.modules:
    %pip install -qqq icechunk "rustytree-xarray>=0.3.0" "xradar>=0.12.0" \
        "zarr>=3.1.2" "s3fs>=2025.5.1" cmweather
    # Fetch the shared helper module so the imports below resolve on Colab.
    !wget -q https://raw.githubusercontent.com/AtmoScale/radar-datatree/main/notebooks/demo_functions.py

In [ ]:
import sys
from pathlib import Path

# Make notebooks/demo_functions.py importable when sphinx-build runs the
# rendered docs page from docs/ (which is a symlink to ../notebooks/).
# Harmless on Colab — the bootstrap above fetches demo_functions.py into
# the kernel CWD, and a path that doesn't exist is silently ignored.
sys.path.insert(0, str(Path("../notebooks").resolve()))

from demo_functions import connect_to_nexrad_arco

# One-line connect to the public KLOT archive on AWS Open Data — anonymous
# S3 reads, no credentials needed. See the helper's docstring for the
# s3_storage / Repository.open / readonly_session boilerplate it wraps.
session = connect_to_nexrad_arco("KLOT")
print("Connected to s3://nexrad-arco/KLOT on branch 'main'")

We can use [`xarray.open_datatree`](https://docs.xarray.dev/en/stable/generated/xarray.open_datatree.html) to explore the radar archive — with `engine="rustytree"`, a Rust-backed xarray DataTree backend recommended for radar-datatree archives. It's a drop-in replacement for the standard `engine="zarr"`, ~10× faster on icechunk repos served from object storage (see [`rustytree-xarray` on PyPI](https://pypi.org/project/rustytree-xarray/)).

In [ ]:
import xarray as xr
import xradar  # noqa: F401  — registers the .xradar accessor

dt = xr.open_datatree(session.store, engine="rustytree", chunks=None)
dt

We can access any of these VCPs using a file-path syntax — for example, `dt["VCP-212/sweep_0"]` — which returns an `xarray.Dataset`. Before slicing anything though, let's see how big the full archive is.

In [ ]:
print(f"datatree size: {dt.nbytes / 1024**4:.2f} TB")

That's almost 100 TB of radar data spanning January 2020 to May 2026 — too much to hold in a single session. Fortunately, we don't have to: we can open only the sweeps or VCPs we care about.

## Opening only what you need

`xarray` and `zarr` let us inspect a dataset's structure by pulling metadata only — no data is read until we ask for it. `rustytree` gives us two ways to slice the archive at open time:

- **`group_filter`** takes a glob and returns a *filtered DataTree* — the matching nodes plus their parent VCP groups, auto-included as ancestors. `"/*/sweep_0"` returns the lowest-elevation cut (`sweep_0`) from every VCP.
- **`group`** takes one exact node path and returns a single *Dataset* via `xr.open_dataset` — the leanest read when you want just one node's arrays.

In [ ]:
dt_sweep0 = xr.open_datatree(
    session.store, engine="rustytree", group_filter="/*/sweep_0"
)

In [ ]:
dt_sweep0

Or pull a **single VCP and sweep** as a plain Dataset with `group`. Here we open **VCP-212** — NEXRAD's severe-weather rapid-scan mode — at its lowest elevation. `open_dataset` returns just that node's arrays: lean and fast, but *without* the coordinates inherited from ancestor nodes (the radar's lat/lon, the `vcp_time` index) — note the short coordinate list below.

In [ ]:
# Pull a single node as a plain Dataset with `group` — the leanest read.
ds_212 = xr.open_dataset(
    session.store,
    engine="rustytree",
    group="/VCP-212/sweep_0",
)

In [ ]:
ds_212

## Plot a polarimetric snapshot

Now that the data is **analysis-ready and cloud-optimized**, plotting is the easy part. Georeferencing needs the coordinates inherited from ancestor nodes — the radar's lat/lon and the `vcp_time` index — which the lean single-Dataset open above leaves out. So we reopen `VCP-212/sweep_0` as a filtered DataTree with `group_filter` and flatten it with `inherit="all_coords"`. Then we grab a single timestamp and georeference it so the polar (azimuth, range) gates land on Cartesian (x, y) axes.

In [ ]:
# Reopen the sweep as a filtered DataTree so we inherit the coordinates
# georeferencing needs (range, azimuth, vcp_time, the radar's lat/lon, …).
dt_212 = xr.open_datatree(
    session.store,
    engine="rustytree",
    group_filter="/VCP-212/sweep_0",
)
scan_ds = dt_212["VCP-212/sweep_0"].to_dataset(inherit="all_coords")

# Snap to the scan closest to the target timestamp. method="nearest" so we
# get the actual VCP cadence — exact-match would raise if the radar wasn't
# scanning at that instant.
scan_at_t = scan_ds.sel(vcp_time="2026-03-10 23:20", method="nearest")

# Add Cartesian x, y, z coordinates derived from (azimuth, range,
# elevation, radar lat/lon). Polar gates now land on Cartesian axes for
# plotting.
scan = scan_at_t.xradar.georeference()

In [ ]:
scan

In [ ]:
from demo_functions import plot_polarimetric_panel

plot_polarimetric_panel(scan);

**What you're looking at.** This is **KLOT** (Chicago, IL) at 2026-03-10 23:20 UTC — a strong convective cell south of the radar. A few things to look for:

- **DBZH** over 50 dBZ marks intense precipitation or hail.
- The **ZDR ↑ + RHOHV ↓** pair (high differential reflectivity, dropped correlation) flags mixed-phase hydrometeors — likely melting or wet hail near the leading edge.
- The **PHIDP** gradient across the storm core scales with rain-path integrated water and is what powers polarimetric rainfall-rate algorithms.

The same five-step pattern — **connect → load → slice → georeference → plot** — works for any timestamp, any VCP, any KLOT scan in the archive.

## Where to next

**Want to reproduce a published figure?**
→ [Notebook 2 — QVP comparison](2.QVP-Workflow-Comparison) reproduces Ryzhkov et al. (2016) Fig. 4 and asserts numerical equivalence between the traditional file-based path and the ARCO streaming path.

**Want to estimate rainfall accumulation?**
→ [Notebook 3 — QPE scaling](3.QPE-Scaling-Benchmark) applies the Marshall–Palmer Z–R relation live for one day, with cluster-recommended templates for 7-day / 30-day / 6-month windows.

**Want to query a different radar or event?**
→ The [quickstart](quickstart) shows the 5-line connection pattern. Swap `"KLOT"` for `"KVNX"` (Oklahoma) to point at a different archive — more radars are added to `nexrad-arco` as they're processed.

**Want to understand the data model?**
→ [About](about) covers the DataTree / Icechunk / Zarr stack and the [AtmoScale](https://atmoscale.ai) parent platform. [Glossary](glossary) defines every radar acronym in one place.

---

*Cite this work:* Ladino-Rincón, A., et al. (2026). *Radar DataTree: A FAIR and Cloud-Native Framework for Scalable Weather Radar Archives.* (Manuscript in preparation.) Earlier preprint: arXiv:2510.24943, [doi:10.48550/arXiv.2510.24943](https://doi.org/10.48550/arXiv.2510.24943).